# 7.9 模型怎麼知道回答結束？

# 第 7 章：從續寫文字到回答問題

前置：第4章的因果模型與第1章的資料對齊。先用一筆不截斷對話，再回讀padding。完整render在tiny_perceptron/data.py；下列程式每節都新建資料，不依賴上一Notebook變量。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：兩種遮罩：看得見，不一定直接考它**

X 的 assistant 格預測 A；PAD 不當答案。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/masks.svg")))

**先想一想：**把EOS從loss去掉，還能要求模型可靠結束嗎？

把EOS當成答案的最後一個訓練token，模型纔有機會學會停止。生成器檢測EOS；另設token上限以防模型永遠不選EOS。

**把直覺寫成數學：**停止條件是EOS或max_new_tokens，不是猜標點就是結尾。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.data import ByteTokenizer, render_chat

tok = ByteTokenizer()
x, y = render_chat([{"role": "user", "content": "Q"}, {"role": "assistant", "content": "A"}])
print("最後目標", y[-1].item(), "EOS", tok.eos_id)
assert y[-1] == tok.eos_id
generated = [tok.encode("A")[0], tok.eos_id, 99]
print("停止前輸出", generated[: generated.index(tok.eos_id)])

**如何讀結果：**這是停止規則例子，不是訓練後的生成質量。

**只改一件事：**只移除EOS，觀察需要靠最大長度停止。
